# NanoQuant · Qwen3.8-27B · 0.55 bpw · Google Colab

Этот ноутбук клонирует [твой форк NanoQuant](https://github.com/rkuznetsovcg/NanoQuant), ставит нужные зависимости, берёт уже подготовленный калибровочный набор из репозитория, запускает профиль `qwen38-0.55-balanced` и сохраняет итоговый `.pt` в Google Drive.

**Как запустить:** в Colab выбери `Среда выполнения → Сменить тип среды выполнения → GPU` и доступный GPU с памятью около 80 ГБ (H100 предпочтителен). Затем выбери `Среда выполнения → Выполнить все` и подтверди доступ к Google Drive. Ячейки проверят, хватает ли памяти и диска, прежде чем скачивать веса модели.

Конфигурация квантования и набор из 128 окон длиной 2 048 токенов берутся из форка. Ноутбук не меняет профиль и не запускает отдельную оценку PPL.

Возобновляющие снимки блоков по умолчанию пишутся на локальный диск Colab ради скорости: Drive медленный для этих многогигабайтных временных файлов. Если сама VM Colab будет уничтожена, локальные снимки пропадут. Для сохранения снимков между VM включи `PERSIST_RESUME_ON_DRIVE` в следующей ячейке; это может заметно увеличить время записи.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rkuznetsovcg/NanoQuant.git"
REPO_BRANCH = "main"
MODEL_ID = "Qwen/Qwen3.8-27B"
PROFILE_RELATIVE_PATH = "configs/qwen38-0.55-balanced.json"
RUN_NAME = "qwen38-27b-0.55-balanced"

# False ускоряет запись промежуточных чекпойнтов. Поставь True до первого запуска,
# если нужно переживать полное уничтожение Colab VM; Drive будет получать крупные снимки блоков.
PERSIST_RESUME_ON_DRIVE = False

# Защита от старта на слишком маленьком runtime. Не снижай пороги, если не уверен в памяти.
MIN_GPU_VRAM_GIB = 70.0
MIN_HOST_RAM_TOTAL_GIB = 70.0
MIN_HOST_RAM_AVAILABLE_GIB = 65.0
MIN_FREE_DISK_GIB = 120.0

ENABLE_FLASHQLA_WHEN_COMPATIBLE = True
DRIVE_MOUNT = Path("/content/drive")
DRIVE_RUN_DIR = DRIVE_MOUNT / "MyDrive" / "NanoQuant" / "runs" / RUN_NAME
REPO_DIR = Path("/content/NanoQuant")
LOCAL_RUN_DIR = Path("/content/nanoquant") / RUN_NAME
LOCAL_OUTPUT = LOCAL_RUN_DIR / "Qwen3.8-27B-NQ-0.55-balanced.pt"


In [ ]:
import os
import re
import shutil
from pathlib import Path

import torch


def gib(value):
    return value / (1024 ** 3)


def read_meminfo_gib(label):
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith(label + ":"):
            return int(line.split()[1]) * 1024 / (1024 ** 3)
    raise RuntimeError(f"Не удалось прочитать {label} из /proc/meminfo")


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU не найден. В Colab выбери Runtime → Change runtime type → GPU и запусти всё снова.")

props = torch.cuda.get_device_properties(0)
gpu_vram_gib = gib(props.total_memory)
host_total_gib = read_meminfo_gib("MemTotal")
host_available_gib = read_meminfo_gib("MemAvailable")
disk_free_gib = gib(shutil.disk_usage("/content").free)
capability = torch.cuda.get_device_capability(0)
cuda_version = torch.version.cuda or "нет CUDA runtime"

def version_tuple(value):
    return tuple(int(part) for part in re.findall(r"\d+", value.split("+", 1)[0])[:3])

flashqla_compatible = (
    ENABLE_FLASHQLA_WHEN_COMPATIBLE
    and capability >= (9, 0)
    and version_tuple(torch.__version__) >= (2, 8)
    and version_tuple(cuda_version) >= (12, 8)
)

print(f"GPU: {props.name} | {gpu_vram_gib:.1f} GiB | compute {capability[0]}.{capability[1]}")
print(f"PyTorch {torch.__version__} | CUDA runtime {cuda_version}")
print(f"RAM: {host_available_gib:.1f} GiB available / {host_total_gib:.1f} GiB total")
print(f"Free /content disk: {disk_free_gib:.1f} GiB")
print(f"FlashQLA install profile: {'eligible' if flashqla_compatible else 'standard Qwen fast kernels'}")

problems = []
if gpu_vram_gib < MIN_GPU_VRAM_GIB:
    problems.append(f"нужно не менее {MIN_GPU_VRAM_GIB:.0f} GiB GPU VRAM")
if host_total_gib < MIN_HOST_RAM_TOTAL_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_TOTAL_GIB:.0f} GiB общей RAM: BF16-веса сначала загружаются в RAM")
if host_available_gib < MIN_HOST_RAM_AVAILABLE_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_AVAILABLE_GIB:.0f} GiB свободной RAM")
if disk_free_gib < MIN_FREE_DISK_GIB:
    problems.append(f"нужно не менее {MIN_FREE_DISK_GIB:.0f} GiB свободного диска под веса и промежуточные блоки")
if problems:
    raise RuntimeError("Этот runtime мал для выбранного запуска: " + "; ".join(problems))


## Подключение Google Drive

Drive нужен для итогового чекпойнта и файла с настройками запуска. Саму модель и временную работу NanoQuant хранит в диске Colab, чтобы не читать и писать веса через сетевую файловую систему Drive.


In [ ]:
from google.colab import drive

DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)
drive.mount(str(DRIVE_MOUNT))
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Итоговые файлы будут сохранены в: {DRIVE_RUN_DIR}")


## Получение форка и калибровочного набора

Набор уже лежит в GitHub-репозитории, поэтому здесь не нужно отдельно скачивать датасеты или собирать их заново.


In [ ]:
import os
import subprocess
import sys


def run_command(command, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    command_env = os.environ.copy() if env is None else env.copy()
    source_dir = REPO_DIR / "src"
    if source_dir.is_dir():
        old_pythonpath = command_env.get("PYTHONPATH", "")
        command_env["PYTHONPATH"] = str(source_dir) + (os.pathsep + old_pythonpath if old_pythonpath else "")
    subprocess.run(command, cwd=cwd, env=command_env, check=True)


if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    run_command(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
    run_command(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

PROFILE_PATH = REPO_DIR / PROFILE_RELATIVE_PATH
CALIBRATION_PATH = REPO_DIR / "data/calibration/generated/qwen3.8-27b-multidomain-128x2048/dataset"
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f"В форке нет профиля: {PROFILE_PATH}")
if not (CALIBRATION_PATH / "dataset_info.json").is_file():
    raise FileNotFoundError(f"В форке нет калибровочного набора: {CALIBRATION_PATH}")

# Keep source imports available even if Colab's editable-install path is not refreshed.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"NanoQuant commit: {GIT_COMMIT}")
print(f"Профиль: {PROFILE_PATH}")
print(f"Калибровка: {CALIBRATION_PATH}")


## Установка CUDA-зависимостей

Для H100 ноутбук сначала пробует совместимый профиль с FlashQLA для Gated DeltaNet; если установка не проходит, он автоматически ставит стандартные быстрые Qwen-ядра и оставляет линейное внимание на FLA. FlashQLA использует тот же BF16-путь модели и включается только на совместимой связке GPU, CUDA и PyTorch.


In [ ]:
import os
import sys


def install_extra(extra):
    command = [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-cache-dir", "-e", f".[{extra}]"]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy()).returncode


if flashqla_compatible:
    status = install_extra("qwen-flashqla")
    if status != 0:
        print("FlashQLA не собрался в этом runtime; включаю резервный путь FLA Triton + kernels.")
        os.environ["FLA_FLASH_QLA"] = "0"
        status = install_extra("qwen-fast,kernels")
else:
    status = install_extra("qwen-fast,kernels")

if status != 0:
    raise RuntimeError("Не удалось установить зависимости NanoQuant. См. вывод pip выше.")
print("Зависимости установлены.")


## Проверка модели, зависимостей и калибровки

Эта ячейка загружает только конфигурацию модели и локальный небольшой калибровочный набор. Полные 27B BF16-веса начнут скачиваться только в следующей ячейке.


In [ ]:
import json
import sys
from datasets import load_from_disk
from transformers import AutoConfig

# The project uses a src/ layout; add it explicitly for this notebook kernel.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from nanoquant.utils.attention_utils import resolve_attention_backend
from nanoquant.utils.load_utils import _check_fast_linear_attention

profile = json.loads(PROFILE_PATH.read_text())
if profile["model_id"] != MODEL_ID:
    raise ValueError(f"Профиль указывает неожиданную модель: {profile['model_id']}")
if profile["bits"] != 0.55 or profile["num_calib_samples"] != 128 or profile["seqlen"] != 2048:
    raise ValueError("Профиль больше не совпадает с запланированным запуском 0.55 bpw / 128 × 2048")

model_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
if model_config.model_type not in {"qwen3_5", "qwen3_5_text"}:
    raise RuntimeError(f"Неожиданный тип модели: {model_config.model_type}")
_check_fast_linear_attention(model_config, required=True)
attention_backend = resolve_attention_backend(model_config, profile.get("attn_implementation", "auto"))

calibration = load_from_disk(str(CALIBRATION_PATH))
if len(calibration) < profile["num_calib_samples"]:
    raise ValueError(f"В датасете только {len(calibration)} строк, нужно {profile['num_calib_samples']}")
first_window = calibration[0]["input_ids"]
if len(first_window) != profile["seqlen"]:
    raise ValueError(f"Длина окна {len(first_window)}, ожидалось {profile['seqlen']}")

LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
resume_path = (DRIVE_RUN_DIR / "resume") if PERSIST_RESUME_ON_DRIVE else (LOCAL_RUN_DIR / "resume")
resume_path.mkdir(parents=True, exist_ok=True)

profile["model_id"] = MODEL_ID
profile["calib_dataset"] = str(CALIBRATION_PATH)
profile["qmodel_path"] = str(LOCAL_OUTPUT)
profile["resume_dir"] = str(resume_path)
profile["attn_implementation"] = "auto"
CONFIG_PATH = LOCAL_RUN_DIR / "qwen38-0.55-balanced-colab.json"
CONFIG_PATH.write_text(json.dumps(profile, indent=2) + "\n")

print(f"Модель: {model_config.model_type}; параметры: {model_config.text_config.num_hidden_layers if hasattr(model_config, 'text_config') else model_config.num_hidden_layers} слоёв")
print(f"Полное внимание: {attention_backend}")
print(f"Калибровка: {len(calibration)} окон; по {len(first_window)} токенов")
print(f"Resume: {resume_path}")
print(f"Промежуточная конфигурация: {CONFIG_PATH}")


## Запуск квантования и сохранение результата

Запуск использует тот же профиль из репозитория. Если процесс прервётся, но Colab VM останется живой, снова выполни последнюю ячейку: NanoQuant продолжит с сохранённого блока. После успешного завершения ноутбук скопирует чекпойнт и метаданные в Drive.

Если появится ошибка о нехватке системной RAM, нужен runtime с большим объёмом **обычной RAM**, даже если у GPU 80 ГБ VRAM. Веса этой модели в BF16 сначала загружаются в системную память.


In [ ]:
import datetime
import json
import shutil

DRIVE_OUTPUT = DRIVE_RUN_DIR / LOCAL_OUTPUT.name
DRIVE_COMPLETE = DRIVE_RUN_DIR / "run_complete.json"
DRIVE_TEMP_OUTPUT = DRIVE_RUN_DIR / (LOCAL_OUTPUT.name + ".uploading")

if DRIVE_COMPLETE.is_file() and DRIVE_OUTPUT.is_file() and DRIVE_OUTPUT.stat().st_size > 0:
    print(f"Готовый запуск уже сохранён: {DRIVE_OUTPUT}")
    print(f"Метка завершения: {DRIVE_COMPLETE}")
else:
    # Если предыдущая запись .pt оборвалась, убираем неполный файл.
    if LOCAL_OUTPUT.exists():
        LOCAL_OUTPUT.unlink()

    run_command([sys.executable, "-m", "nanoquant.main", str(CONFIG_PATH)], cwd=REPO_DIR)
    if not LOCAL_OUTPUT.is_file() or LOCAL_OUTPUT.stat().st_size == 0:
        raise RuntimeError("Команда завершилась без итогового чекпойнта")

    shutil.copy2(LOCAL_OUTPUT, DRIVE_TEMP_OUTPUT)
    if DRIVE_TEMP_OUTPUT.stat().st_size != LOCAL_OUTPUT.stat().st_size:
        DRIVE_TEMP_OUTPUT.unlink(missing_ok=True)
        raise IOError("Размер файла после копирования в Drive не совпал с исходным")
    DRIVE_TEMP_OUTPUT.replace(DRIVE_OUTPUT)

    run_metadata = {
        "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "model_id": MODEL_ID,
        "profile": PROFILE_RELATIVE_PATH,
        "bits": 0.55,
        "calibration_windows": 128,
        "sequence_length": 2048,
        "attention_backend": attention_backend,
        "gpu": props.name,
        "gpu_vram_gib": round(gpu_vram_gib, 2),
        "pytorch": torch.__version__,
        "cuda_runtime": cuda_version,
        "nanoquant_commit": GIT_COMMIT,
        "checkpoint_file": DRIVE_OUTPUT.name,
        "checkpoint_size_bytes": DRIVE_OUTPUT.stat().st_size,
    }
    temp_metadata = DRIVE_RUN_DIR / "run_complete.json.tmp"
    temp_metadata.write_text(json.dumps(run_metadata, indent=2) + "\n")
    temp_metadata.replace(DRIVE_COMPLETE)

    print("Квантование завершено.")
    print(f"Чекпойнт: {DRIVE_OUTPUT}")
    print(f"Метаданные: {DRIVE_COMPLETE}")


### Что будет в Google Drive

В папке `MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced/` появятся:

- `Qwen3.8-27B-NQ-0.55-balanced.pt` — итоговый чекпойнт;
- `run_complete.json` — профиль, ревизия кода, GPU и параметры запуска.

Открыть этот ноутбук позднее можно из форка: [NanoQuant Colab notebook](https://colab.research.google.com/github/rkuznetsovcg/NanoQuant/blob/main/notebooks/qwen38_0.55_colab.ipynb).
